## Quality of Care Indicators

Compute district-year quality-of-care indicators from DHIS2 outliers-imputed routine data.

Indicators:
- testing_rate = TEST / SUSP
- treatment_rate = MALTREAT / CONF
- case_fatality_rate = MALDTH / MALADM
- prop_adm_malaria = MALADM / ALLADM
- prop_malaria_deaths = MALDTH / ALLDTH
- non_malaria_all_cause_outpatients = ALLOUT (absolute)
- presumed_cases = PRES (absolute)

Stock-out indicators are not implemented yet (on hold, NMDR data pending).

In [ ]:
# Preliminaries
options(scipen=999)

ROOT_PATH <- "~/workspace"
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
CODE_PATH <- file.path(ROOT_PATH, "code")
DATA_PATH <- file.path(ROOT_PATH, "data")
OUTPUT_DATA_PATH <- file.path(DATA_PATH, "dhis2", "quality_of_care")
FIGURES_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care", "reporting", "outputs", "figures")

dir.create(OUTPUT_DATA_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_PATH, recursive = TRUE, showWarnings = FALSE)

source(file.path(CODE_PATH, "snt_utils.r"))
required_packages <- c("jsonlite", "data.table", "arrow", "sf", "ggplot2", "glue", "reticulate", "RColorBrewer", "writexl")
install_and_load(required_packages)

Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
openhexa <- reticulate::import("openhexa.sdk")

config_json <- jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_config.json"))
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
DHIS2_FORMATTED_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
OUTLIERS_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_OUTLIERS_IMPUTATION

In [ ]:
# Fallback parameters for local/dev execution
if (!exists("outlier_imputation_method")) {
  outlier_imputation_method <- "mean"
}
if (!exists("data_action")) {
  data_action <- "imputed"
}

allowed_methods <- c("mean", "median", "iqr", "trend", "mg-partial", "mg-complete")
if (!(outlier_imputation_method %in% allowed_methods)) {
  stop(glue::glue("Invalid outlier_imputation_method: {outlier_imputation_method}. Allowed: {paste(allowed_methods, collapse=', ')}"))
}

allowed_actions <- c("imputed", "removed")
if (!(data_action %in% allowed_actions)) {
  stop(glue::glue("Invalid data_action: {data_action}. Allowed: {paste(allowed_actions, collapse=', ')}"))
}

# Magic Glasses uses different file naming convention
is_mg <- outlier_imputation_method %in% c("mg-partial", "mg-complete")

if (is_mg) {
  # For Magic Glasses, use formatted routine data (no imputed files exist)
  # Note: Magic Glasses produces outlier flags, not imputed routine files
  # We use the base formatted routine data
  routine_filename <- glue::glue("{COUNTRY_CODE}_routine.parquet")
  log_msg(glue::glue("Magic Glasses method selected. Loading base routine file: {routine_filename}"))
  log_msg("[WARNING] Magic Glasses does not produce imputed routine files. Using base formatted routine data.")
  routine <- get_latest_dataset_file_in_memory(DHIS2_FORMATTED_DATASET, routine_filename)
} else {
  # Standard methods: use outliers-imputed files
  routine_filename <- glue::glue("{COUNTRY_CODE}_routine_outliers-{outlier_imputation_method}_{data_action}.parquet")
  log_msg(glue::glue("Loading routine file from DHIS2 outliers dataset: {routine_filename}"))
  routine <- get_latest_dataset_file_in_memory(OUTLIERS_DATASET, routine_filename)
}

shapes <- get_latest_dataset_file_in_memory(DHIS2_FORMATTED_DATASET, paste0(COUNTRY_CODE, "_shapes.geojson"))

setDT(routine)

# Core required columns (must exist)
core_cols <- c("ADM2_ID", "YEAR")
core_missing <- setdiff(core_cols, names(routine))
if (length(core_missing) > 0) {
  stop(glue::glue("Missing core required columns in routine data: {paste(core_missing, collapse=', ')}"))
}

# Optional indicator columns (will be checked and handled gracefully)
indicator_cols <- c("TEST", "SUSP", "MALTREAT", "CONF", "MALDTH", "MALADM", "ALLADM", "ALLDTH", "ALLOUT", "PRES")
available_cols <- intersect(indicator_cols, names(routine))
missing_cols <- setdiff(indicator_cols, names(routine))

if (length(missing_cols) > 0) {
  log_msg(glue::glue("[WARNING] Some indicator columns are missing: {paste(missing_cols, collapse=', ')}. These indicators will not be calculated."), level = "warning")
}

# Convert available numeric columns
num_cols <- intersect(available_cols, names(routine))
if (length(num_cols) > 0) {
  routine[, (num_cols) := lapply(.SD, function(x) as.numeric(x)), .SDcols = num_cols]
}
routine[, YEAR := as.integer(YEAR)]
routine[, ADM2_ID := as.character(ADM2_ID)]

# Aggregate available columns only using lapply
if (length(available_cols) > 0) {
  qoc <- routine[, lapply(.SD, function(x) sum(x, na.rm = TRUE)), 
                 .SDcols = available_cols, 
                 by = .(ADM2_ID, YEAR)]
} else {
  # If no indicator columns available, create empty structure
  qoc <- routine[, .(ADM2_ID, YEAR)]
  qoc <- unique(qoc)
}

# Calculate indicators only if required columns are available
if ("TEST" %in% names(qoc) && "SUSP" %in% names(qoc)) {
  qoc[, testing_rate := fifelse(SUSP > 0, TEST / SUSP, NA_real_)]
} else {
  log_msg("[WARNING] Cannot calculate testing_rate: missing TEST or SUSP columns", level = "warning")
}

if ("MALTREAT" %in% names(qoc) && "CONF" %in% names(qoc)) {
  qoc[, treatment_rate := fifelse(CONF > 0, MALTREAT / CONF, NA_real_)]
} else {
  log_msg("[WARNING] Cannot calculate treatment_rate: missing MALTREAT or CONF columns", level = "warning")
}

if ("MALDTH" %in% names(qoc) && "MALADM" %in% names(qoc)) {
  qoc[, case_fatality_rate := fifelse(MALADM > 0, MALDTH / MALADM, NA_real_)]
} else {
  log_msg("[WARNING] Cannot calculate case_fatality_rate: missing MALDTH or MALADM columns", level = "warning")
}

if ("MALADM" %in% names(qoc) && "ALLADM" %in% names(qoc)) {
  qoc[, prop_adm_malaria := fifelse(ALLADM > 0, MALADM / ALLADM, NA_real_)]
} else {
  log_msg("[WARNING] Cannot calculate prop_adm_malaria: missing MALADM or ALLADM columns", level = "warning")
}

if ("MALDTH" %in% names(qoc) && "ALLDTH" %in% names(qoc)) {
  qoc[, prop_malaria_deaths := fifelse(ALLDTH > 0, MALDTH / ALLDTH, NA_real_)]
} else {
  log_msg("[WARNING] Cannot calculate prop_malaria_deaths: missing MALDTH or ALLDTH columns", level = "warning")
}

if ("ALLOUT" %in% names(qoc)) {
  qoc[, non_malaria_all_cause_outpatients := ALLOUT]
} else {
  log_msg("[WARNING] Cannot calculate non_malaria_all_cause_outpatients: missing ALLOUT column", level = "warning")
}

if ("PRES" %in% names(qoc)) {
  qoc[, presumed_cases := PRES]
} else {
  log_msg("[WARNING] Cannot calculate presumed_cases: missing PRES column", level = "warning")
}

shapes_dt <- as.data.table(sf::st_drop_geometry(shapes))
if ("ADM2_ID" %in% names(shapes_dt) && "ADM2_NAME" %in% names(shapes_dt)) {
  shapes_dt[, ADM2_ID := as.character(ADM2_ID)]
  qoc <- merge(qoc, unique(shapes_dt[, .(ADM2_ID, ADM2_NAME)]), by = "ADM2_ID", all.x = TRUE)
}

out_parquet <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_{outlier_imputation_method}_{data_action}.parquet"))
out_csv <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_{outlier_imputation_method}_{data_action}.csv"))
out_xlsx <- file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_{outlier_imputation_method}_{data_action}.xlsx"))

arrow::write_parquet(qoc, out_parquet)
data.table::fwrite(qoc, out_csv)
writexl::write_xlsx(list(quality_of_care = as.data.frame(qoc)), out_xlsx)

log_msg(glue::glue("Saved outputs: {out_parquet}, {out_csv}, {out_xlsx}"))

In [ ]:
# Yearly maps by ADM2
shapes$ADM2_ID <- as.character(shapes$ADM2_ID)
qoc$ADM2_ID <- as.character(qoc$ADM2_ID)

plot_yearly_map <- function(df, sf_shapes, value_col, title_prefix, filename_prefix, is_rate = TRUE) {
  years <- sort(unique(df$YEAR))
  for (yr in years) {
    df_y <- df[YEAR == yr]
    map_df <- merge(sf_shapes, df_y, by = "ADM2_ID", all.x = TRUE)

    p <- ggplot(map_df)

    if (is_rate) {
      map_df$cat <- cut(
        map_df[[value_col]],
        breaks = c(-Inf, 0, 0.2, 0.4, 0.6, 0.8, 1.0, Inf),
        labels = c("<0", "0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0", ">1.0"),
        include.lowest = TRUE
      )
      p <- p + geom_sf(aes(fill = cat), color = "grey60", size = 0.1) +
        scale_fill_brewer(palette = "YlOrRd", na.value = "white", drop = FALSE)
    } else {
      vals <- map_df[[value_col]]
      finite_vals <- vals[is.finite(vals)]
      if (length(finite_vals) > 4) {
        br <- unique(as.numeric(quantile(finite_vals, probs = seq(0, 1, 0.2), na.rm = TRUE)))
        if (length(br) < 2) {
          map_df$cat <- as.factor("all")
        } else {
          map_df$cat <- cut(vals, breaks = br, include.lowest = TRUE)
        }
      } else {
        map_df$cat <- as.factor(vals)
      }
      p <- p + geom_sf(aes(fill = cat), color = "grey60", size = 0.1) +
        scale_fill_brewer(palette = "Blues", na.value = "white", drop = FALSE)
    }

    p <- p +
      theme_void() +
      labs(
        title = paste0(title_prefix, " - ", yr),
        fill = value_col,
        caption = "Source: SNT DHIS2 outliers-imputed routine data"
      ) +
      theme(
        legend.position = "bottom",
        plot.title = element_text(face = "bold", size = 12)
      )

    out_png <- file.path(FIGURES_PATH, glue::glue("{filename_prefix}_{yr}.png"))
    ggsave(out_png, plot = p, width = 9, height = 7, dpi = 300, bg = "white")
  }
}

# Plot only indicators that were calculated (columns exist)
if ("testing_rate" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "testing_rate", "Testing rate (TEST / SUSP)", "testing_rate", TRUE)
}
if ("treatment_rate" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "treatment_rate", "Treatment rate (MALTREAT / CONF)", "treatment_rate", TRUE)
}
if ("case_fatality_rate" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "case_fatality_rate", "In-hospital case fatality rate (MALDTH / MALADM)", "case_fatality_rate", TRUE)
}
if ("prop_adm_malaria" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "prop_adm_malaria", "Proportion admitted for malaria (MALADM / ALLADM)", "prop_adm_malaria", TRUE)
}
if ("prop_malaria_deaths" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "prop_malaria_deaths", "Proportion of malaria deaths (MALDTH / ALLDTH)", "prop_malaria_deaths", TRUE)
}
if ("non_malaria_all_cause_outpatients" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "non_malaria_all_cause_outpatients", "Non-malaria all-cause outpatients (ALLOUT)", "allout", FALSE)
}
if ("presumed_cases" %in% names(qoc)) {
  plot_yearly_map(qoc, shapes, "presumed_cases", "Presumed cases (PRES)", "presumed_cases", FALSE)
}

log_msg(glue::glue("Saved yearly maps in: {FIGURES_PATH}"))